In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Setup Paths
# Note: In Jupyter, '__file__' doesn't exist, so we use os.getcwd()
project_dir = Path(os.getcwd())
raw_csv_path = project_dir / "fbref_ultimate_2425.csv"
clean_csv_path = project_dir / "fbref_cleaned_2425.csv"

print(f"📂 Looking for data at: {raw_csv_path}")

if not raw_csv_path.exists():
    print("❌ Error: Raw CSV not found. Did you run the scraper?")
else:
    print("✅ Found raw data file.")

📂 Looking for data at: c:\Users\DELL\Documents\Moneyball\fbref_ultimate_2425.csv
✅ Found raw data file.


In [2]:
# Load with header=[0, 1] because FBref gives us 2 rows of headers
df = pd.read_csv(raw_csv_path, header=[0, 1], index_col=[0, 1, 2, 3])

# Show the first 3 rows
print(f"Data Shape: {df.shape}")
df.head(3)

Data Shape: (2854, 161)


nation  \
                                              Unnamed: 4_level_1   
league             season team    player                           
ENG-Premier League 2425   Arsenal Ben White                  ENG   
                                  Bukayo Saka                ENG   
                                  David Raya                 ESP   

                                                             pos  \
                                              Unnamed: 5_level_1   
league             season team    player                           
ENG-Premier League 2425   Arsenal Ben White                   DF   
                                  Bukayo Saka              FW,MF   
                                  David Raya                  GK   

                                                             age  \
                                              Unnamed: 6_level_1   
league             season team    player                           
ENG-Premier League 2425   Arsenal Ben White                 26.0   
                                  Bukayo Saka               22.0   
                                  David Raya                28.0   

                                                            born Playing Time  \
                                              Unnamed: 7_level_1           MP   
league             season team    player                                        
ENG-Premier League 2425   Arsenal Ben White               1997.0           17   
                                  Bukayo Saka             2001.0           25   
                                  David Raya              1995.0           38   

                                                                 Performance  \
                                              Starts   Min   90s         Gls   
league             season team    player                                       
ENG-Premier League 2425   Arsenal Ben White       13  1198  13.3           0   
                                  Bukayo Saka     20  1729  19.2           6   
                                  David Raya      38  3420  38.0           0   

                                                   ... Passes Goal Kicks  \
                                              Ast  ... AvgLen        Att   
league             season team    player           ...                     
ENG-Premier League 2425   Arsenal Ben White     2  ...    NaN        NaN   
                                  Bukayo Saka  10  ...    NaN        NaN   
                                  David Raya    0  ...   29.9      147.0   

                                                             Crosses        \
                                              Launch% AvgLen     Opp   Stp   
league             season team    player                                     
ENG-Premier League 2425   Arsenal Ben White       NaN    NaN     NaN   NaN   
                                  Bukayo Saka     NaN    NaN     NaN   NaN   
                                  David Raya     68.0   49.1   402.0  53.0   

                                                    Sweeper                  
                                               Stp%    #OPA #OPA/90 AvgDist  
league             season team    player                                     
ENG-Premier League 2425   Arsenal Ben White     NaN     NaN     NaN     NaN  
                                  Bukayo Saka   NaN     NaN     NaN     NaN  
                                  David Raya   13.2    66.0    1.74    17.2  

[3 rows x 161 columns]

In [3]:
# Flatten columns
new_columns = [f"{col[0]}_{col[1]}" for col in df.columns]
df.columns = new_columns

print("✅ Columns flattened.")
# Check the first 10 column names to see if they look cleaner
df.columns[:10]

✅ Columns flattened.


Index(['nation_Unnamed: 4_level_1', 'pos_Unnamed: 5_level_1',
       'age_Unnamed: 6_level_1', 'born_Unnamed: 7_level_1', 'Playing Time_MP',
       'Playing Time_Starts', 'Playing Time_Min', 'Playing Time_90s',
       'Performance_Gls', 'Performance_Ast'],
      dtype='object')

In [4]:
# Create a dictionary to hold our rename mapping
rename_map = {}

# 1. FIND 'MINUTES'
# Look for columns containing 'Min' and ('Standard' or 'Playing')
min_cols = [c for c in df.columns if 'Min' in c and ('Standard' in c or 'Playing' in c or 'Time' in c)]
print(f"Potential 'Minutes' columns found: {min_cols}")

# We pick the first valid one
if min_cols:
    rename_map[min_cols[0]] = 'Minutes'

# 2. FIND 'POSITION'
# Look for columns with 'pos' but IGNORE 'Possession'
pos_cols = [c for c in df.columns if 'pos' in c.lower() and 'possession' not in c.lower()]
print(f"Potential 'Position' columns found: {pos_cols}")

if pos_cols:
    rename_map[pos_cols[0]] = 'Position'

# 3. FIND 'AGE'
age_cols = [c for c in df.columns if 'age' in c.lower() and 'percentage' not in c.lower()]
print(f"Potential 'Age' columns found: {age_cols}")

if age_cols:
    rename_map[age_cols[0]] = 'Age'

print("\n--- FINAL MAPPING ---")
print(rename_map)

Potential 'Minutes' columns found: ['Playing Time_Min']
Potential 'Position' columns found: ['pos_Unnamed: 5_level_1', 'pos_drop_Unnamed: 38_level_1', 'pos_drop_Unnamed: 60_level_1', 'pos_drop_Unnamed: 88_level_1', 'pos_drop_Unnamed: 109_level_1', 'pos_drop_Unnamed: 136_level_1']
Potential 'Age' columns found: ['age_Unnamed: 6_level_1', 'age_drop_Unnamed: 39_level_1', 'age_drop_Unnamed: 61_level_1', 'age_drop_Unnamed: 89_level_1', 'age_drop_Unnamed: 110_level_1', 'age_drop_Unnamed: 137_level_1']

--- FINAL MAPPING ---
{'Playing Time_Min': 'Minutes', 'pos_Unnamed: 5_level_1': 'Position', 'age_Unnamed: 6_level_1': 'Age'}


In [5]:
# Apply the renaming
df = df.rename(columns=rename_map)

# Verify strictly
required_cols = ['Minutes', 'Position', 'Age']
missing = [c for c in required_cols if c not in df.columns]

if missing:
    print(f"❌ CRITICAL ERROR: We are still missing: {missing}")
else:
    print("✅ Success! Critical columns are mapped.")
    print(df[['Minutes', 'Position', 'Age']].head())

✅ Success! Critical columns are mapped.
                                                 Minutes Position   Age
league             season team    player                               
ENG-Premier League 2425   Arsenal Ben White         1198       DF  26.0
                                  Bukayo Saka       1729    FW,MF  22.0
                                  David Raya        3420       GK  28.0
                                  Declan Rice       2825       MF  25.0
                                  Ethan Nwaneri      895    FW,MF  17.0


In [6]:
# Check for nulls in critical columns
null_check = df[['Minutes', 'Position', 'Age']].isnull().sum()
print("Missing values in key columns:")
print(null_check)

# Optional: Fill empty stats with 0 (NaN goals usually means 0 goals)
df = df.fillna(0)
print("\n✅ Filled remaining NaNs with 0.")

Missing values in key columns:
Minutes     0
Position    0
Age         2
dtype: int64

✅ Filled remaining NaNs with 0.


In [7]:
# Filter: Keep players with > 500 minutes
original_count = len(df)
df_clean = df[df['Minutes'] > 500].copy()
new_count = len(df_clean)

print(f"Original Player Count: {original_count}")
print(f"Filtered Player Count: {new_count}")
print(f"Dropped {original_count - new_count} players (too few minutes).")

# Save to CSV (Clean, flat, ready for ML)
df_clean.to_csv(clean_csv_path)
print(f"\n💾 Saved clean database to: {clean_csv_path}")

Original Player Count: 2854
Filtered Player Count: 1927
Dropped 927 players (too few minutes).

💾 Saved clean database to: c:\Users\DELL\Documents\Moneyball\fbref_cleaned_2425.csv


In [9]:
# 1. Check the structure (Shows column names and if data is missing)
print("--- DATAFRAME INFO ---")
df_clean.info()

# 2. Check the Index (Should be League, Season, Team, Player)
print("\n--- INDEX STRUCTURE ---")
print(df_clean.index.names)

print("\n--- SAMPLE OF CRITICAL COLUMNS ---")
# Use 'player' and 'team' because your index is properly named!
print(df_clean.reset_index()[['player', 'team', 'Position', 'Minutes', 'Age']].head(5))

--- DATAFRAME INFO ---
<class 'pandas.core.frame.DataFrame'>
MultiIndex: 1927 entries, ('ENG-Premier League', np.int64(2425), 'Arsenal', 'Ben White') to ('ITA-Serie A', np.int64(2425), 'Venezia', 'Ridgeciano Haps')
Columns: 161 entries, nation_Unnamed: 4_level_1 to Sweeper_AvgDist
dtypes: float64(78), int64(71), object(12)
memory usage: 2.5+ MB

--- INDEX STRUCTURE ---
['league', 'season', 'team', 'player']

--- SAMPLE OF CRITICAL COLUMNS ---
          player     team Position  Minutes   Age
0      Ben White  Arsenal       DF     1198  26.0
1    Bukayo Saka  Arsenal    FW,MF     1729  22.0
2     David Raya  Arsenal       GK     3420  28.0
3    Declan Rice  Arsenal       MF     2825  25.0
4  Ethan Nwaneri  Arsenal    FW,MF      895  17.0
